In [ ]:
# ============================================================
# 1. CARGA DE LIBRERÍAS
# ============================================================

!pip install fastai -q

from fastai.text.all import *
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [2]:
# ============================================================
# 2. CARGA Y PREPROCESAMIENTO DEL DATASET
# ============================================================

# Mismo dataset IMDB
path = untar_data(URLs.IMDB)

# Mismo esquema de dls para compararlo con LSTM/GRU desde cero
dls = TextDataLoaders.from_folder(
    path,
    valid='test',
    bs=32,
    seq_len=72
)

In [3]:
# ============================================================
# 3. CARGA DEL MODELO PREENTRENADO AWD-LSTM (ULMFiT)
# ============================================================

# Creamos el learner con AWD-LSTM + métricas estándar
learn = text_classifier_learner(
    dls,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=[accuracy]
)

# Cargar encoder preentrenado (ULMFiT oficial)
learn.load_encoder('finetuned')

FileNotFoundError: [Errno 2] No such file or directory: '/root/.fastai/data/imdb/models/finetuned.pth'

In [ ]:
# ============================================================
# 4. ENTRENAMIENTO DEL MODELO (FINE-TUNING)
# ============================================================

# Entrenamiento cabeza del clasificador
learn.fit_one_cycle(1, 2e-2)

# Descongelar todo para fine tuning
learn.unfreeze()

# Entrenamiento completo del modelo
learn.fit_one_cycle(3, 2e-3)



In [ ]:
# ============================================================
# 5. MÉTRICAS GLOBALES
# ============================================================

val_loss, val_acc = learn.validate()
print(f"\nValidation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
# ============================================================
# 6. MATRIZ DE CONFUSIÓN
# ============================================================

preds, targs = learn.get_preds()
cm = confusion_matrix(targs, preds.argmax(dim=1))

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión - ULMFiT (AWD-LSTM)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

In [ ]:

# ============================================================
# 7. GRÁFICOS DE LOSS Y ACCURACY (FIX OFICIAL PARA FASTAI)
# ============================================================

# Fastai guarda historiales en learn.recorder
losses = learn.recorder.losses
train_losses = learn.recorder.losses
valid_losses = [v[0] for v in learn.recorder.values]  # primer valor: loss
accuracies = [v[1] for v in learn.recorder.values]    # segundo valor: accuracy

# ---- LOSS ----
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot([len(train_losses)//len(valid_losses)*i for i in range(len(valid_losses))],
         valid_losses, marker='o', label="Valid Loss")
plt.xlabel("Iteraciones")
plt.ylabel("Loss")
plt.title("Loss - Entrenamiento ULMFiT")
plt.legend()
plt.show()

# ---- ACCURACY ----
plt.figure(figsize=(8,5))
plt.plot(range(len(accuracies)), accuracies, marker='o')
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy - Entrenamiento ULMFiT")
plt.show()